# Project Setup

In [ ]:
# Import Necessary Libraries #
import os
import cv2
import glob
import torch
import warnings
import numpy as np
import seaborn as sns
import torch.nn as nn
from tqdm.auto import tqdm
import torch.optim as optim
import albumentations as alb
from torch.amp import autocast
import torchvision.models as tv
import matplotlib.pyplot as plt
from torch.amp import GradScaler
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression

In [ ]:
warnings.filterwarnings('ignore')

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
scaler = GradScaler()

print("Device: ", device)

# Define Constants & Functions

In [ ]:
# Define Constants #
# Dataset paths
AI4MARS_IMG = "ai4mars-dataset-merged-0.1/msl/images/edr"
AI4MARS_TRAIN = "ai4mars-dataset-merged-0.1/msl/labels/train"
AI4MARS_TEST = "ai4mars-dataset-merged-0.1/msl/labels/test/masked-gold-min2-100agree"

FREIFOR_TRAIN = "freiburg_forest_annotated/train"

# AI4MARS Label key (Slimmer constants - For np.isin() compatibility)
AI4M_TRAVERSABLE = (0, 1)                                       # We'll consider soil & bedrock as traversable terrain
AI4M_HAZARDOUS = (2, 3)                                         # We'll consider sand & big rock as hazardous terrain

# Freiburg Forest Label key (Slimmer constants - For np.isin() compatibility)
FF_TRAVERSABLE = (170,170,170)                                  # We'll consider Road as traversable terrain
FF_HAZARDOUS = ((0,255,0), (51,102,102), (0,60,0), (0,0,0))     # We'll consider Grass, Vegetation, Tree, & Obstacle as hazardous terrain

In [ ]:
# Define Utility Functions & Classes #
# Define a function to center-crop image arrays
def center_crop(arr, size=320):                 # size refers to square pixel dimension of cropped array
    h, w = arr.shape[:2]
    y0, x0 = (h - size) // 2, (w - size) // 2   # Finds the top-left corner of the cropped image

    return arr[y0:y0+size, x0:x0+size]

# Define a function to load mask images from the AI4MARS dataset
def load_ai4m_mask(path):
    single = cv2.imread(path)[:,:,0]  # Reduces mask to single byte by isolating channel 0 (values can be {0,1,2,3,255})
    output = np.full_like(single, 255, dtype=np.uint8)      # Start with all output pixels set to 255 (ignore)

    # Assign traversable pixels
    output[np.isin(single, AI4M_TRAVERSABLE)] = 0
    # Assign non-traversable pixels
    output[np.isin(single, AI4M_HAZARDOUS)] = 1

    return output

# Define a function to load mask images from the Freiburg Forest dataset
def load_ff_mask(path):
    mask = cv2.imread(path)
    output = np.full(mask.shape[:2], 255, np.uint8)         # Start with all output pixels set to 255 (ignore)

    traversable = np.all(mask == FF_TRAVERSABLE, axis=-1)   # Define pixels matching Road channels as safe
    output[traversable] = 0                                 # Assign traversable pixels

    # Define pixels matching hazardous channels as non_traversable
    non_traversable = np.zeros_like(traversable)
    for colour in FF_HAZARDOUS:
        non_traversable |= np.all(mask == colour, axis=-1)
    output[non_traversable] = 1                             # Assign non-traversable pixels

    return output

# Function to build image pair (raw, masked) for AI4MARS dataset
def ai4m_pairs(mask_directory):
    mask_files = glob.glob(os.path.join(mask_directory, "*.png"))

    if mask_directory == AI4MARS_TRAIN:
        return [(os.path.join(AI4MARS_IMG, os.path.basename(p)[:-4] + ".JPG"), p) for p in mask_files]
    elif mask_directory == AI4MARS_TEST:
        return [(os.path.join(AI4MARS_IMG, os.path.basename(p)[:-11] + ".JPG"), p) for p in mask_files]

# Function to build image pair (raw, masked) for Freiburg Forest dataset
def ff_pairs(mask_directory):
    return [(os.path.join(mask_directory, "rgb", f), os.path.join(mask_directory, "GT_color", f[:-11] + "mask" + ".png"))
            for f in os.listdir(os.path.join(mask_directory, "rgb")) if f.endswith(".jpg")]

# Define function to extract features and labels from image,mask pairs
def feat_label_extract(img, mask):
    mean_rgb = img.reshape(-1,3).mean(axis=0)               # Flatten the image (320x320x3 -> 102400x3) and get the mean of each channel
    grey = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)            # Convert image to greyscale
    std_grey = grey.std()                                   # Compute standard deviation of grey shades (high std = strong texture/contrast)
    edge_den = cv2.Canny(grey, 50, 150).mean() / 255.       # Run Canny edge detector and divide mean by 255 to get edge pixels (edge-dense sections generally hazardous terrain)
    hazard_ratio = (mask == 1).mean()                       # Calculate ratio of non-traversable pixels
    label = int(hazard_ratio > 0.5)                         # Label pixel patch 1 if more than 50% hazardous pixels and 0 otherwise

    return np.hstack([mean_rgb, std_grey, edge_den]), label # Return feature vector and binary label

# Define function to build feature and label arrays for baseline regression
def build_Xy(pairs, desc="Building Pairs"):
    X, y = [], []
    num_safe = num_hazard = 0

    for img_p, msk_p in tqdm(pairs, desc=desc, leave=True):
        image = cv2.cvtColor(cv2.imread(img_p), cv2.COLOR_BGR2RGB)
        mask = load_ai4m_mask(msk_p) if "ai4mars" in msk_p else load_ff_mask(msk_p)
        image, mask = center_crop(image), center_crop(mask)                         # Crop images to uniform size
        feature, label = feat_label_extract(image, mask)                            # Extract feature vector and label

        X.append(feature)
        y.append(label)

        if label == 1:
            num_hazard += 1
        else:
            num_safe += 1

    return np.asarray(X), np.asarray(y), (num_safe / max(num_hazard, 1))            # Return arrays of feature vectors and labels

# Define a class for a 320x320 image dataset (to be used with PyTorch CNNs)
class ImageDS(Dataset):
    def __init__(self, pairs, augment=False, size=320):
        self.pairs = pairs
        self.augment = augment
        self.size = size
        self.augmentations = alb.Compose([
            alb.HorizontalFlip(),
            alb.Rotate(15, border_mode=cv2.BORDER_REFLECT),
            alb.ColorJitter(0.2, 0.2, 0.1, 0.05)
        ])

    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, index):
        img_p, msk_p = self.pairs[index]
        image = cv2.cvtColor(cv2.imread(img_p), cv2.COLOR_BGR2RGB)
        mask = load_ai4m_mask(msk_p) if "ai4mars" in msk_p else load_ff_mask(msk_p)
        image, mask = center_crop(image, self.size), center_crop(mask, self.size)

        if self.augment:
            augment = self.augmentations(image=image, mask=mask)
            image, mask = augment["image"], augment["mask"]

        x = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.
        y = torch.tensor([(mask == 1).mean() > 0.5], dtype=torch.float32)

        return x, y

# Define a Mini-CNN model
class MiniCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, 1)
        )
    
    def forward(self, x):
        return self.layers(x).squeeze(1)
    
# Define a MobileNet V2 model
class MobileNetV2(nn.Module):
    def __init__(self):
        super().__init__()

        base = tv.mobilenet_v2(weights="IMAGENET1K_V1")

        for parameter in base.parameters():
            parameter.requires_grad_(False)
        
        self.features = base.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(1280, 1)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        
        return self.fc(x).squeeze(1)
    
# Define function to plot a confusion matrix
def plot_conf_mat(y_true, y_pred, title, ax):
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["safe","hazard"], yticklabels=["safe","hazard"], cbar=False, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

# Build Data Pairs

In [ ]:
# Build Train/Test Pairs #
mars_train = ai4m_pairs(AI4MARS_TRAIN)
mars_test  = ai4m_pairs(AI4MARS_TEST)

earth_train = ff_pairs(FREIFOR_TRAIN)

combo_train = mars_train + earth_train

print(f"Mars train/test : {len(mars_train):,}/{len(mars_test):,}")
print(f"Earth train : {len(earth_train):,}")

print(f"\nCombined train/test : {len(combo_train):,}/{len(mars_test):,}")

# Feature Extraction & Logistic Regression

In [ ]:
# Extract Feature,Label Datasets for Regression #
print("Extracting Mars features…")
X_mars_train, y_mars_train, mars_class_ratio = build_Xy(mars_train, desc="Building Mars Training Pairs")
X_mars_test, y_mars_test, _ = build_Xy(mars_test, desc="Building Mars Testing Pairs")

print("Extracting Combo (Mars + Earth) features…")
X_combo_train, y_combo_train, combo_class_ratio = build_Xy(combo_train, desc="Building Combined Training Pairs")

In [ ]:
# Define a Function to Train & Evaluate Basic Logistic Regression #
def logreg(X_train, y_train, X_test, y_test, label=""):
    classification = LogisticRegression(max_iter=2000, class_weight='balanced')
    classification.fit(X_train, y_train)

    y_pred = classification.predict(X_test)
    balanced_accuracy = balanced_accuracy_score(y_test, y_pred)

    print(f"\n~~~ {label} Logistic Regression ~~~")
    print(classification_report(y_test, y_pred, target_names=["safe","hazard"], digits=3))
    
    return classification, y_pred, balanced_accuracy

In [ ]:
# Train & Evaluate Two Logistic Regression Baselines #
# Mars-only baseline
mars_logreg_clf, logreg_mars_pred, mars_logreg_ba = logreg(X_mars_train, y_mars_train, X_mars_test, y_mars_test, "Mars-only")

# Mars + Earth baseline
combo_logreg_clf, logreg_combo_pred, combo_logreg_ba = logreg(X_combo_train, y_combo_train, X_mars_test, y_mars_test, "Mars + Earth")

# Train Mini-CNN and MobileNetV2

In [ ]:
# Define a Function to Train & Evaluate CNN Models #
def runCNN(model, train_pairs, class_ratio, name, epochs=15, lr=1e-3, batch_size=16, early_stop=3):
    val_dl = DataLoader(ImageDS(mars_test, augment=False), batch_size=32)

    if isinstance(model, MobileNetV2):
        train_dl = DataLoader(ImageDS(train_pairs, augment=True, size=224), batch_size=batch_size, shuffle=True, pin_memory=True)
    else:   # Mini-CNN
        train_dl = DataLoader(ImageDS(train_pairs, augment=True), batch_size=batch_size, shuffle=True)

    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([class_ratio], device=device))
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_balanced_accuracy = patience = 0
    ba_hist = []

    for epoch in range(epochs):
        # Train
        model.train()

        for x_batch, y_batch in tqdm(train_dl, desc=f"{name} training | Epoch {epoch}", leave=True):
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad(set_to_none=True)
            
            with autocast(device_type="cuda"):
                loss = criterion(model(x_batch), y_batch.squeeze(1))

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        
        # Validate
        model.eval()
        tp = tn = fp = fn = 0

        with torch.inference_mode():
            for x_batch, y_batch in tqdm(val_dl, desc=f"{name} validating | Epoch {epoch}", leave=False):
                pred = (torch.sigmoid(model(x_batch.to(device))) > 0.5).cpu()

                tp += ((pred == 1) & (y_batch == 1)).sum()
                tn += ((pred == 0) & (y_batch == 0)).sum()
                fp += ((pred == 1) & (y_batch == 0)).sum()
                fn += ((pred == 0) & (y_batch == 1)).sum()

        balanced_accuracy = ((tp / (tp + fn + 1e-7)) + (tn / (tn + fp + 1e-7))) * 0.5
        ba_hist.append(balanced_accuracy)

        if balanced_accuracy > best_balanced_accuracy:
            best_balanced_accuracy, patience = balanced_accuracy, 0
            torch.save(model.state_dict(), f"{name}_best.pt")
        else:
            patience += 1

            if patience == early_stop:
                break

    # Evaluate
    model.load_state_dict(torch.load(f"{name}_best.pt"))

    y_true, y_pred = [], []

    with torch.no_grad():
        for x_batch, y_batch in val_dl:
            y_pred += (torch.sigmoid(model(x_batch.to(device))) > 0.5).cpu().tolist()
            y_true += y_batch.tolist()

    balanced_accuracy = balanced_accuracy_score(y_true, y_pred)

    print(f"\n~~~ {'Mars' if 'Mars' in name else 'Mars + Earth'} {'Mini-CNN' if 'MiniCNN' in name else 'UNet-MobileNet'} ~~~")
    print(classification_report(y_true, y_pred, target_names=["safe","hazard"], digits=3))

    return model, y_pred, balanced_accuracy, ba_hist

In [ ]:
# Train & Evaluate Two Mini-CNN Models #
# Mars-only model
minicnn_best_mars_model, minicnn_mars_pred, mars_minicnn_ba, mars_hist_cnn = runCNN(MiniCNN(), mars_train, mars_class_ratio, "MiniCNN_Mars")

# Mars + Earth model
minicnn_best_combo_model, minicnn_combo_pred, combo_minicnn_ba, combo_hist_cnn = runCNN(MiniCNN(), combo_train, combo_class_ratio, "MiniCNN_Combo")

In [ ]:
# Train & Evaluate Two MobileNet Models #
# Mars-only model
mobilenet_best_mars_model, mobilenet_mars_pred, mars_mobilenet_ba, mars_hist_mn = runCNN(MobileNetV2(), mars_train, mars_class_ratio, "MobileNet_Mars", batch_size=8)

# Mars + Earth model
mobilenet_best_combo_model, mobilenet_combo_pred, combo_mobilenet_ba, combo_hist_mn = runCNN(MobileNetV2(), combo_train, combo_class_ratio, "MobileNet_Combo", batch_size=8)

# {Code to Use Pre-Trained Models}

In [ ]:
# # Mini-CNN & MobileNet models have already been run and best models have already been isolated
# val_dl = DataLoader(ImageDS(mars_test, augment=False), batch_size=32)

# minicnn_best_mars_model = MiniCNN().to(device)
# minicnn_best_combo_model = MiniCNN().to(device)
# mobilenet_best_mars_model = MobileNetV2().to(device)
# mobilenet_best_combo_model = MobileNetV2().to(device)

# minicnn_best_mars_model.load_state_dict(torch.load("MiniCNN_Mars_best.pt"))
# minicnn_best_combo_model.load_state_dict(torch.load("MiniCNN_Combo_best.pt"))
# mobilenet_best_mars_model.load_state_dict(torch.load("MobileNet_Mars_best.pt"))
# mobilenet_best_combo_model.load_state_dict(torch.load("MobileNet_Combo_best.pt"))

# # Mars
# y_true, minicnn_mars_pred, minicnn_combo_pred, mobilenet_mars_pred, mobilenet_combo_pred = [], [], [], [], []

# with torch.no_grad():
#     for x_batch, y_batch in val_dl:
#         minicnn_mars_pred += (torch.sigmoid(minicnn_best_mars_model(x_batch.to(device))) > 0.5).cpu().tolist()
#         minicnn_combo_pred += (torch.sigmoid(minicnn_best_combo_model(x_batch.to(device))) > 0.5).cpu().tolist()
#         mobilenet_mars_pred += (torch.sigmoid(mobilenet_best_mars_model(x_batch.to(device))) > 0.5).cpu().tolist()
#         mobilenet_combo_pred += (torch.sigmoid(mobilenet_best_combo_model(x_batch.to(device))) > 0.5).cpu().tolist()
#         y_true += y_batch.tolist()

# mars_minicnn_ba = balanced_accuracy_score(y_true, minicnn_mars_pred)
# combo_minicnn_ba = balanced_accuracy_score(y_true, minicnn_combo_pred)
# mars_mobilenet_ba = balanced_accuracy_score(y_true, mobilenet_mars_pred)
# combo_mobilenet_ba = balanced_accuracy_score(y_true, mobilenet_combo_pred)

# print(f"\n~~~ Mars Mini-CNN ~~~")
# print(classification_report(y_true, minicnn_mars_pred, target_names=["safe","hazard"], digits=3))
# print(f"\n~~~ Mars + Earth Mini-CNN ~~~")
# print(classification_report(y_true, minicnn_combo_pred, target_names=["safe","hazard"], digits=3))
# print(f"\n~~~ Mars MobileNet ~~~")
# print(classification_report(y_true, mobilenet_mars_pred, target_names=["safe","hazard"], digits=3))
# print(f"\n~~~ Mars + Earth MobileNet ~~~")
# print(classification_report(y_true, mobilenet_combo_pred, target_names=["safe","hazard"], digits=3))

# Metrics & Charts

In [ ]:
# Print Summary Table of Balanced Accuracies #
print("===== Balanced Accuracy on Mars Test Dataset =====")
print(f"{'Model':20} {'Mars-only':>10} {'Mars+Earth':>10}")
print(f"{'Logistic Regression':20} {mars_logreg_ba:10.3f} {combo_logreg_ba:10.3f}")
print(f"{'Mini-CNN':20} {mars_minicnn_ba:10.3f} {combo_minicnn_ba:10.3f}")
print(f"{'MobileNet-V2':20} {mars_mobilenet_ba:10.3f} {combo_mobilenet_ba:10.3f}")

In [ ]:
# Plot Confusion Matrices for each Model & Dataset #
fig, axs = plt.subplots(3, 2, figsize=(8,8))

plot_conf_mat(y_mars_test, logreg_mars_pred, "LogReg - Mars", axs[0,0])
plot_conf_mat(y_mars_test, logreg_combo_pred, "LogReg - Mars+Earth", axs[0,1])
plot_conf_mat(y_mars_test, minicnn_mars_pred, "MiniCNN - Mars", axs[1,0])
plot_conf_mat(y_mars_test, minicnn_combo_pred, "MiniCNN - Mars+Earth", axs[1,1])
plot_conf_mat(y_mars_test, mobilenet_mars_pred, "MobileNet - Mars", axs[2,0])
plot_conf_mat(y_mars_test, mobilenet_combo_pred, "MobileNet - Mars+Earth", axs[2,1])
plt.tight_layout()

plt.show()

In [ ]:
# Plot Bar Chart of Balanced Accuracy Scores for each Model & Dataset #
labels = ["LogReg", "MiniCNN", "MobileNet"]
mars_scores  = [mars_logreg_ba,  mars_minicnn_ba,  mars_mobilenet_ba]
combo_scores = [combo_logreg_ba, combo_minicnn_ba, combo_mobilenet_ba]

x = np.arange(len(labels))
w = 0.35

plt.figure(figsize=(6,4))

plt.bar(x-w/2, mars_scores,  width=w, label="Mars-only")
plt.bar(x+w/2, combo_scores, width=w, label="Mars+Earth")
plt.xticks(x, labels)
plt.ylabel("Balanced accuracy")
plt.ylim(0,1)
plt.legend()
plt.title("Model comparison on Mars test set")
plt.tight_layout()

plt.show()

In [ ]:
# Plot Mini-CNN Learning Curves #
plt.plot(mars_hist_cnn, label="MiniCNN Mars")
plt.plot(combo_hist_cnn, label="MiniCNN Mars + Earth")
plt.xlabel("Epoch")
plt.ylabel("Balanced accuracy")
plt.legend()
plt.title("Learning curves")

plt.show()

In [ ]:
# Plot MobileNet Learning Curves #
plt.plot(mars_hist_mn, label="MobileNet Mars")
plt.plot(combo_hist_mn, label="MobileNet Mars + Earth")
plt.xlabel("Epoch")
plt.ylabel("Balanced accuracy")
plt.legend()
plt.title("Learning curves")

plt.show()